# Afrihealth Disease Surveillance - EDA

In [3]:
# Import libraries and initial setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [19]:
# Load data

geo_types = {
    "urban_rural": "category"
}

fac_types = {
    "facility_type": "category",
    "facility_level": "category",
    "operation_status": "category",
    "established_year": "category"
}

reports_types = {
   "disease": "category",
   "age_group": "category",
   "gender": "category" 
}

geo = pd.read_csv('../data/raw/geographical.csv', dtype=geo_types)
fac = pd.read_csv('../data/raw/facilities.csv', dtype=fac_types)
weather = pd.read_csv('../data/raw/weather.csv')
reports = pd.read_csv('../data/raw/reports.csv', dtype=reports_types)

# Convert dates
weather['date'] = pd.to_datetime(weather['date'])
reports['report_date'] = pd.to_datetime(reports['report_date'])
reports['case_date'] = pd.to_datetime(reports['case_date'])

#### Data Quality Checks

In [ ]:
# Scan datasets to check for quality issues.
def quality_check(df, name):
    # Check for missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)

    if missing.sum() > 0:
        for col in missing[missing > 0].index:
            print(f"{col}: {missing[col]} missing values, making ({missing_pct[col]}%) in {name} dataset.")
    else:
        print(f'No missing values in {name} dataset')
    
    # Duplicates
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        print(f'Duplicates: {dup_count} rows in {name} dataset.')
    else:
        print(f'No duplicates in {name} dataset')

### Disease Overview

In [43]:
total_cases = reports['new_cases'].sum()
total_deaths = reports['deaths'].sum()

fatality_rate = (total_deaths / total_cases) * 100 if total_cases > 0 else 0

disease_summary = reports.groupby('disease').agg({
    "new_cases": "sum",
    "deaths": "sum",
    "recoveries": "sum"
}).round(0)

disease_summary['fatality_rate_pct'] = ((disease_summary['deaths'] / disease_summary['new_cases']) * 100).round(2)

reports_geo = reports.merge(geo[['geography_id', 'country_code', 'country_name']], how='left', on='geography_id')
country_summary = reports_geo.groupby('country_name').agg({
    "new_cases": "sum",
    "deaths": "sum",
    "recoveries": "sum"
})
country_summary['fatality_rate_pct'] = ((country_summary['deaths'] / country_summary['new_cases']) * 100).round(2)

#### Time Trends